In [ ]:
from IPython.display import display

from src.phase3.curation import (
    PHASE3_CURATION_DATA_DIR,
    build_phase3_all_experiment_specs,
    build_phase3_combined_experiment_specs,
    build_phase3_individual_experiment_specs,
    prepare_phase3_experiment_dataset,
)
from src.curation_evaluation import (
    PHASE3_EVALUATION_RESULTS_ROOT,
    run_phase3_experiment,
    select_best_phase3_individual_options,
    select_best_phase3_spec,
    summarize_phase3_experiment_specs,
)

FORCE_REBUILD_DATASETS = False
FORCE_SCORE = False
FORCE_TRAIN = False

print(f"CSV output dir: data/{PHASE3_CURATION_DATA_DIR}")
print(f"Training output root: results/<architecture>/{PHASE3_EVALUATION_RESULTS_ROOT}")

## 1. Definir experimentos individuales

Se prueban por separado `top%`, `dedup` y `quality` sobre cada dataset de partida de fase 2.

In [ ]:
individual_specs = build_phase3_individual_experiment_specs()

print(f"Individual datasets: {len(individual_specs)}")
for spec in individual_specs:
    print(f"- {spec.descriptor}: {spec.output_csv}")

## 2. Crear CSVs individuales

Los CSVs se guardan en `data/phase3test`. Si ya existen y `FORCE_REBUILD_DATASETS=False`, se reutilizan.

In [ ]:
prepared_individual = []

for index, spec in enumerate(individual_specs, start=1):
    print(f"[{index}/{len(individual_specs)}] Preparing {spec.descriptor}")
    result = prepare_phase3_experiment_dataset(
        spec,
        force_rebuild=FORCE_REBUILD_DATASETS,
        force_score=FORCE_SCORE,
    )
    prepared_individual.append(result)
    print(f"    output_csv={result['output_csv']} | reused={result['reused']}")

## 3. Entrenar individuales y mostrar metricas

Cada experimento usa 2 seeds. Los plots se guardan por el entrenamiento, pero no se muestran.

In [ ]:
individual_metric_tables = {}

for index, spec in enumerate(individual_specs, start=1):
    print("=" * 100)
    print(f"[{index}/{len(individual_specs)}] Training {spec.descriptor}")
    metrics_df = run_phase3_experiment(
        spec,
        force_rebuild_dataset=False,
        force_score=FORCE_SCORE,
        force_train=FORCE_TRAIN,
    )
    individual_metric_tables[spec.descriptor] = metrics_df
    display(metrics_df)

## 4. Resumen individual y seleccion de mejores parametros

La seleccion usa `macro_f1`; si la diferencia es menor que `0.005`, desempata por `mcc`.

In [ ]:
individual_summary = summarize_phase3_experiment_specs(individual_specs)
display(individual_summary)

best_options = select_best_phase3_individual_options(individual_summary)
print("Best top fraction:", best_options["best_top_fraction"])
print("Best dedup mode:", best_options["best_dedup_mode"])
print("Best quality mode:", best_options["best_quality_mode"])

display(best_options["best_top_row"].to_frame().T)
display(best_options["best_dedup_row"].to_frame().T)
display(best_options["best_quality_row"].to_frame().T)

## 5. Definir combinaciones finales

Con los mejores valores encontrados, se prueban solo combinaciones de 2 o 3 condiciones: `top+dedup`, `top+quality`, `dedup+quality`, `top+dedup+quality`.

In [ ]:
combined_specs = build_phase3_combined_experiment_specs(
    best_top_fraction=best_options["best_top_fraction"],
    best_dedup_mode=best_options["best_dedup_mode"],
    best_quality_mode=best_options["best_quality_mode"],
)

print(f"Combined datasets: {len(combined_specs)}")
for spec in combined_specs:
    print(f"- {spec.descriptor}: {spec.output_csv}")

## 6. Crear CSVs combinados

In [ ]:
prepared_combined = []

for index, spec in enumerate(combined_specs, start=1):
    print(f"[{index}/{len(combined_specs)}] Preparing {spec.descriptor}")
    result = prepare_phase3_experiment_dataset(
        spec,
        force_rebuild=FORCE_REBUILD_DATASETS,
        force_score=FORCE_SCORE,
    )
    prepared_combined.append(result)
    print(f"    output_csv={result['output_csv']} | reused={result['reused']}")

## 7. Entrenar combinados y mostrar metricas

In [ ]:
combined_metric_tables = {}

for index, spec in enumerate(combined_specs, start=1):
    print("=" * 100)
    print(f"[{index}/{len(combined_specs)}] Training {spec.descriptor}")
    metrics_df = run_phase3_experiment(
        spec,
        force_rebuild_dataset=False,
        force_score=FORCE_SCORE,
        force_train=FORCE_TRAIN,
    )
    combined_metric_tables[spec.descriptor] = metrics_df
    display(metrics_df)

## 8. Tabla final general

Tabla sin clases, ordenada por `macro_f1` y `mcc`.

In [ ]:
all_specs = build_phase3_all_experiment_specs(
    best_top_fraction=best_options["best_top_fraction"],
    best_dedup_mode=best_options["best_dedup_mode"],
    best_quality_mode=best_options["best_quality_mode"],
)

final_summary = summarize_phase3_experiment_specs(all_specs)
display(final_summary)

best_final = select_best_phase3_spec(final_summary)
print("Best final experiment:")
display(best_final.to_frame().T)